# 15 - SVM-heavy blend (0.75 SVM + 0.25 CatBoost, iter-imputed)

**What this is, honestly:** a lottery ticket, not a validated improvement. We exhausted
the real levers (see CLAUDE.md S2/S9): model family, class_weight, thresholds,
multiple/pre-scale imputation, semi-supervised, temporal context, and imputation quality
(eog_burst_index is already recovered at R2=0.874). True quality is ~0.855 and that is the
data's ceiling. The 50/50 SVM+CatBoost ensemble already *lost* on the LB (0.84736).

This blend tilts heavily toward the SVM (0.75) so it stays close to the 0.855 anchor while
borrowing a little of CatBoost's decorrelated signal (0.25). Its expected *private*-LB gain
is ~zero; we build it only so you can A/B it on the public board with spare submissions.
Keep `svm_iterimpute_submission.csv` (0.85456) as the final pick regardless.

## Step 0 - Imports and data

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, numpy as np, pandas as pd
from sklearn.experimental import enable_iterative_imputer   # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier

TRAIN_PATH, TEST_PATH, OUT_DIR = '../train.csv', '../test.csv', '../outputs'
train = pd.read_csv(TRAIN_PATH); test = pd.read_csv(TEST_PATH)
FEATURES = [c for c in train.columns if c not in ('id','sleep_stage')]
X, y, Xtest = train[FEATURES], train['sleep_stage'].to_numpy(), test[FEATURES]
W_SVM = 0.75   # blend weight on the SVM (rest goes to CatBoost)
print('train', train.shape, '| test', test.shape, '| SVM weight', W_SVM)

## Step 1 - The two iter-imputed base models (same tuned params as notebook 11)
SVM uses `probability=True` so we can average class probabilities; CatBoost outputs them
natively. Both consume identical iterative-imputed features.

In [ ]:
def make_imputer():
    return IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42)

def make_svm():
    return make_pipeline(make_imputer(), StandardScaler(),
        SVC(kernel='rbf', C=12, gamma=0.012, probability=True, random_state=42))

def make_catboost():
    return make_pipeline(make_imputer(),
        CatBoostClassifier(loss_function='MultiClass', iterations=600, depth=7,
                           learning_rate=0.04, l2_leaf_reg=3,
                           random_state=42, verbose=0, thread_count=-1))

## Step 2 - (Optional) CV check of the blend weight (slow ~10 min)
For the record: compare SVM-alone vs the 0.75/0.25 blend on OOF probabilities across
seeds. Expect the gain to sit *inside* the +/-0.008 fold noise - that is exactly why this
is a gamble, not an upgrade. Skip straight to Step 3 to just build the submission.

In [ ]:
from scipy import stats
SEEDS = [42, 2024, 7]
svm_fold, bl_fold = [], []
for seed in SEEDS:
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    oof_s = cross_val_predict(make_svm(),      X, y, cv=cv, method='predict_proba', n_jobs=5)
    oof_c = cross_val_predict(make_catboost(), X, y, cv=cv, method='predict_proba', n_jobs=5)
    oof_b = W_SVM*oof_s + (1-W_SVM)*oof_c
    for _, idx in cv.split(X, y):
        svm_fold.append(f1_score(y[idx], oof_s[idx].argmax(1), average='macro'))
        bl_fold.append( f1_score(y[idx], oof_b[idx].argmax(1), average='macro'))
    print('seed %-4d  SVM=%.4f  blend=%.4f'
          % (seed, f1_score(y, oof_s.argmax(1), average='macro'),
                   f1_score(y, oof_b.argmax(1), average='macro')), flush=True)
svm_fold, bl_fold = np.array(svm_fold), np.array(bl_fold)
g = bl_fold - svm_fold
print('\nmean gain (blend - SVM) = %+.4f +/- %.4f over %d folds' % (g.mean(), g.std(), len(g)))
print('blend wins %d/%d folds   paired p = %.2e' % ((g>0).sum(), len(g), stats.ttest_rel(bl_fold, svm_fold).pvalue))

## Step 3 - Build the submission (self-contained)
Refit both models on all training rows, blend test probabilities 0.75/0.25, argmax.

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
svm = make_svm().fit(X, y)
cat = make_catboost().fit(X, y)
proba = W_SVM*svm.predict_proba(Xtest) + (1-W_SVM)*np.asarray(cat.predict_proba(Xtest))
pred = proba.argmax(1).astype(int)
sub = pd.DataFrame({'id': test['id'], 'sleep_stage': pred})
path = os.path.join(OUT_DIR, 'svm_heavy_blend_submission.csv'); sub.to_csv(path, index=False)
print('wrote', path, sub.shape)
print('predicted class distribution:', dict(sub.sleep_stage.value_counts().sort_index()))
sub.head()

## Takeaways
- This is a deliberately conservative blend (75% SVM) so it can't drift far from the 0.855
  anchor. If Step 2's gain is inside +/-0.008 (it will be), treat any public-LB bump as
  sampling noise that won't survive to the private board.
- **Final pick stays `svm_iterimpute_submission.csv` (0.85456).** Use this blend only as a
  spare-submission experiment, not your graded selection.